# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rupamsingh0907-beep/Flyintern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1. What one row means.** The raw fact table has one row per day x client x content item. For Lane 2 my decision grain is one row per page (one `content_hash_id` within a client) for March 2026, with the daily rows combined.

**2. Tables.** `fact_content_daily_performance` for daily search performance, joined to `dim_content` on `content_hash_id` for content age.

**3. Time window.** Month 2026-03. Features use 1-15 March. The label uses 16-31 March, so features always come before the label.

**4. What I rank.** A review priority for pages, using a proxy label: `declined` = 1 if a page's impressions in the second half of March are below 80% of its first-half impressions. This is a proxy (a short-window drop), not a proven loss.

**5. Deliberately excluded.** The `_sample` table, because it holds the last month and would let me peek at the future. Also any product-computed flags or scores, because they are decisions, not observable signals.

In [1]:
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
token = userdata.get("HF_TOKEN")   # saved in Colab Secrets (key icon), notebook access on
con.sql(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{token}')")

con.sql("SELECT * FROM glob('hf://datasets/FlyRank/internship-warehouse/**')").show(max_rows=50)

┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                  file                                                  │
│                                                varchar                                                 │
├────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ hf://datasets/FlyRank/internship-warehouse/.gitattributes                                              │
│ hf://datasets/FlyRank/internship-warehouse/README.md                                                   │
│ hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/dim_content.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet │
│ hf://datasets/FlyRank/internship-wa

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')"
DIM  = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 1").show()
con.sql(f"DESCRIBE SELECT * FROM {DIM} LIMIT 1").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE march AS
SELECT * FROM {FACT}
WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
con.sql("""
SELECT
  COUNT(*) AS total_rows,
  (SELECT COUNT(*) FROM (SELECT DISTINCT report_date, client_hash_id, content_hash_id FROM march)) AS distinct_keys
FROM march
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┐
│ total_rows │ distinct_keys │
│   int64    │     int64     │
├────────────┼───────────────┤
│    9841378 │       9841378 │
└────────────┴───────────────┘



If total_rows equals distinct_keys, one row really is one day x client x content, as I said.

In [5]:
con.sql("""
SELECT COUNT(*) AS rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
       COUNT(DISTINCT client_hash_id) AS clients, COUNT(DISTINCT content_hash_id) AS pages
FROM march
""").show()

┌─────────┬────────────┬────────────┬─────────┬────────┐
│  rows   │ first_day  │  last_day  │ clients │ pages  │
│  int64  │    date    │    date    │  int64  │ int64  │
├─────────┼────────────┼────────────┼─────────┼────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │      55 │ 331437 │
└─────────┴────────────┴────────────┴─────────┴────────┘



This is my slice: how many rows, which dates, and how many clients and pages.

In [6]:
con.sql("""
SELECT COUNT(*) AS rows_total,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4
FROM march
""").show()

┌────────────┬───────────────┐
│ rows_total │ rows_with_ga4 │
│   int64    │     int64     │
├────────────┼───────────────┤
│    9841378 │        413966 │
└────────────┴───────────────┘



Rows before a client’s GA4 tracking started are search-only. Missing GA4 values there mean “not tracked yet,” not “no traffic.” Only this many rows survive when I require GA4.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

My slice is a single month (March 2026), so it cannot show seasonality, and a 15-day vs 16-day split is a short window for judging decline. The panel is also unbalanced: clients began tracking at different times, so some pages have less history than others. The label is a proxy for a drop, not proof that a page needs work.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.